## Ocean Cubes Visualization: Local Cluster Zarr v3 Store

Displays a local **Zarr v3 GeoZarr** store (consolidated `zarr.json` with `spatial:bbox` and `proj:code`).
Convert a Zarr v2 store first:

```bash
python convert_zarr.py ~/bucket/input_v2.zarr ~/bucket/output_v3.zarr          # EPSG read from the store
python convert_zarr.py ~/bucket/input_v2.zarr ~/bucket/output_v3.zarr --epsg 4326  # if it is not
```


In [ ]:
# ==============================================================================
# 0. Start CORS & Local Zarr Proxy (works locally and on EOXHub / JupyterHub)
# ==============================================================================
# - Remote CloudFerro S3 GeoZarrs need CORS headers on HEAD requests.
# - Local cluster Zarr v2/v3 stores (e.g. `bucket/mediterranean_fronts_v3.zarr`)
#   cannot be served via Jupyter's `/files/` route because `ContentsManager`
#   blocks hidden dot-files (`.zmetadata`, `.zgroup`, `.zarray`, `.zattrs`).
# - `cors_proxy.py` handles both: upstream S3 paths (`/...`) and cluster-local
#   files (`/local/...`, with dot-files + HTTP Range/HEAD support).
# - On EOXHub, browser requests reach `127.0.0.1:8010` inside the pod through
#   `jupyter-server-proxy` at `<HUB_ORIGIN><JUPYTERHUB_SERVICE_PREFIX>proxy/8010`.

import http.client
import json
import os
import socket
import subprocess
import sys
import time

PROXY_PORT = 8010
HUB_ORIGIN = os.environ.get("HUB_ORIGIN", "https://jupyterhub.hub-otc-sc.eox.at").rstrip("/")
service_prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX")

# Serve `/local/...` from the Jupyter Server root / $HOME (matching Jupyter's `/files/` root)
# so notebooks inside subfolders (e.g. `~/ocean_hackathon/4_Visualizations`) can still read `~/bucket/...`.
_home = os.path.abspath(os.path.expanduser("~"))
_cwd = os.path.abspath(os.getcwd())
LOCAL_ROOT = os.path.abspath(
    os.environ.get("JUPYTER_SERVER_ROOT")
    or (_home if os.path.commonpath([_home, _cwd]) == _home else _cwd)
)

if service_prefix:
    PROXY_BASE = f"{HUB_ORIGIN}/{service_prefix.strip('/')}/proxy/{PROXY_PORT}"
else:
    PROXY_BASE = f"http://localhost:{PROXY_PORT}"


def _check_proxy_health(port: int):
    try:
        conn = http.client.HTTPConnection("127.0.0.1", port, timeout=3)
        conn.request("GET", "/healthz")
        resp = conn.getresponse()
        body = resp.read().decode("utf-8", errors="replace")
        conn.close()
        if resp.status == 200:
            data = json.loads(body)
            if "local_root" in data:
                return data
    except Exception:
        pass
    return None


with socket.socket() as s:
    port_in_use = s.connect_ex(("127.0.0.1", PROXY_PORT)) == 0

health = _check_proxy_health(PROXY_PORT) if port_in_use else None

# Restart cors_proxy.py if an older instance lacks `/local/` or was started with a different `--local-root`
if port_in_use and (
    health is None or os.path.abspath(health.get("local_root", "")) != LOCAL_ROOT
):
    if "proxy" in globals() and proxy.poll() is None:
        proxy.terminate()
        proxy.wait(timeout=5)
    else:
        subprocess.run(
            ["pkill", "-f", f"cors_proxy.py.*--port {PROXY_PORT}"],
            check=False,
        )
    for _ in range(20):
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", PROXY_PORT)) != 0:
                port_in_use = False
                health = None
                break
        time.sleep(0.1)

if not port_in_use:
    proxy = subprocess.Popen(
        [
            sys.executable,
            "cors_proxy.py",
            "--port",
            str(PROXY_PORT),
            "--local-root",
            LOCAL_ROOT,
        ]
    )
    for _ in range(30):
        health = _check_proxy_health(PROXY_PORT)
        if health is not None:
            break
        time.sleep(0.1)
    print(f"Started CORS & Local Zarr proxy on 127.0.0.1:{PROXY_PORT} (pid {proxy.pid})")
elif health is not None:
    print(f"Proxy already running on 127.0.0.1:{PROXY_PORT}")
else:
    print(
        f"WARNING: Port {PROXY_PORT} is occupied by another process. "
        "Stop that process or change PROXY_PORT above."
    )

# --- Diagnostic Checks ---
if health:
    print(f"  [OK] Internal health check : {health}")
try:
    conn = http.client.HTTPConnection("127.0.0.1", PROXY_PORT, timeout=5)
    conn.request("HEAD", "/EarthCODE/OSCAssets/med_cubes/ocean-med-biophysics.zarr/.zmetadata")
    up_resp = conn.getresponse()
    up_resp.read()
    conn.close()
    print(f"  [OK] Upstream S3 HEAD test : HTTP {up_resp.status} (CORS header: {up_resp.getheader('Access-Control-Allow-Origin')})")
except Exception as exc:
    print(f"  [WARN] Upstream S3 check failed: {exc}")

print(f"\nBrowser Proxy Base URL: {PROXY_BASE}")
if service_prefix:
    print(f"Test EOXHub browser routing by opening: {PROXY_BASE}/healthz")

In [ ]:
# ==============================================================================
# 1. Local Cluster Zarr v3 GeoZarr Store (via /local/ proxy route)
# ==============================================================================
# This cell only accepts EOxElements-compatible Zarr v3 stores: consolidated
# `zarr.json` with `spatial:bbox` and `proj:code` root attributes.
# Convert a Zarr v2 store first with:
#     python convert_zarr.py ~/bucket/input_v2.zarr ~/bucket/output_v3.zarr [--epsg 4326]

import importlib
import json
import os
from pathlib import Path
import ipyeoxelements
import ipyeoxelements.generated

# Reload module if EOxTimecontrol template was already patched in a prior run
if "eox-timecontrol-date" in ipyeoxelements.EOxTimecontrol._esm_template:
    importlib.reload(ipyeoxelements.generated)
    importlib.reload(ipyeoxelements)

from ipyeoxelements import EOxMap, EOxLayercontrol, EOxTimecontrol
from ipywidgets import HBox, Layout, VBox

# Path to the Zarr v3 store (relative to $HOME / Jupyter root, the current folder, or absolute)
local_zarr_rel = "bucket/mediterranean_fronts_v3.zarr"
# Variable shown first; None picks the first one. All variables can be switched in the layer control.
variable = None

_home = os.path.abspath(os.path.expanduser("~"))
_cwd = os.path.abspath(os.getcwd())
_local_root = globals().get("LOCAL_ROOT") or os.path.abspath(
    os.environ.get("JUPYTER_SERVER_ROOT")
    or (_home if os.path.commonpath([_home, _cwd]) == _home else _cwd)
)


def _resolve_local_zarr(zarr_path: str, local_root: str) -> tuple[Path, str]:
    """Find the store in cwd, its parent folders, LOCAL_ROOT or $HOME; return (path, URL path under /local/)."""
    root_abs = os.path.abspath(local_root)
    zarr_path = os.path.expanduser(zarr_path)
    rel_clean = zarr_path.lstrip("/")

    search_bases = []
    cur = os.path.abspath(os.getcwd())
    while True:
        search_bases.append(cur)
        if cur == root_abs or os.path.dirname(cur) == cur:
            break
        cur = os.path.dirname(cur)
    for extra in (root_abs, _home):
        if extra not in search_bases:
            search_bases.append(extra)

    candidates = []
    for base in search_bases:
        cand = os.path.abspath(zarr_path if os.path.isabs(zarr_path) else os.path.join(base, rel_clean))
        if cand not in candidates:
            candidates.append(cand)

    for cand in candidates:
        if os.path.isdir(cand):
            if os.path.commonpath([root_abs, cand]) != root_abs:
                raise ValueError(
                    f"Found '{cand}', but the proxy only serves files under '{root_abs}'. "
                    "Move the store there or set LOCAL_ROOT in the proxy cell to a parent folder."
                )
            print(f"  [OK] Resolved local store: {cand}")
            return Path(cand), os.path.relpath(cand, root_abs).replace(os.sep, "/")

    raise FileNotFoundError(
        f"Zarr store '{zarr_path}' not found. Checked:\n" + "\n".join(f"  - {c}" for c in candidates)
    )


def _read_geozarr_v3(store_path: Path) -> tuple[dict, dict]:
    """Validate an EOxElements-compatible Zarr v3 store; return (root attributes, {variable: array metadata})."""
    hint = f"Convert it with:  python convert_zarr.py {store_path} <output.zarr> [--epsg CODE]"
    root_json = store_path / "zarr.json"
    if not root_json.is_file():
        if (store_path / ".zgroup").is_file() or (store_path / ".zmetadata").is_file():
            raise ValueError(f"'{store_path}' is a Zarr v2 store; only Zarr v3 is supported here.\n{hint}")
        raise ValueError(f"'{store_path}' has no zarr.json, so it is not a Zarr v3 store.\n{hint}")

    root = json.loads(root_json.read_text())
    attrs = root.get("attributes", {})
    meta = (root.get("consolidated_metadata") or {}).get("metadata")
    missing = [k for k in ("spatial:bbox", "proj:code") if k not in attrs]
    if root.get("zarr_format") != 3 or root.get("node_type") != "group":
        raise ValueError(f"'{store_path}' root is not a Zarr v3 group.\n{hint}")
    if missing or not meta:
        problems = ([f"missing root attributes {missing}"] if missing else []) + (
            ["no consolidated metadata"] if not meta else []
        )
        raise ValueError(f"'{store_path}' is not EOxElements-compatible ({', '.join(problems)}).\n{hint}")

    y_dim, x_dim = attrs.get("spatial:dimensions", [None, None])
    variables = {}
    for name, m in meta.items():
        dims = m.get("dimension_names") or []
        if m.get("node_type") != "array" or "/" in name or len(dims) < 2:
            continue
        if y_dim and (y_dim not in dims or x_dim not in dims):
            continue
        variables[name] = m
    if not variables:
        raise ValueError(f"'{store_path}' has no 2-D (or higher) data variables to display.")
    print(
        f"  [OK] Zarr v3 GeoZarr | {attrs['proj:code']} | bbox {attrs['spatial:bbox']} | "
        f"variables: {list(variables)}"
    )
    return attrs, variables


resolved_store_path, resolved_url_rel = _resolve_local_zarr(local_zarr_rel, _local_root)
store_attrs, store_variables = _read_geozarr_v3(resolved_store_path)

if variable is None:
    variable = next(iter(store_variables))
elif variable not in store_variables:
    raise ValueError(f"Variable '{variable}' is not in the store. Choose one of: {list(store_variables)}")

if store_attrs["proj:code"] not in ("EPSG:4326", "EPSG:3857"):
    print(
        f"  [WARN] {store_attrs['proj:code']} is not built into OpenLayers; the layer may not display "
        "unless that projection is registered on the map (see polar.ipynb)."
    )

# Verify running proxy's local_root matches _local_root
if "_check_proxy_health" in globals():
    _h = _check_proxy_health(PROXY_PORT)
    if _h and os.path.abspath(_h.get("local_root", "")) != _local_root:
        print(
            f"  [WARN] Running proxy serves '{_h.get('local_root')}' instead of '{_local_root}'. "
            "Re-run the proxy cell first so `/local/` can reach this store!"
        )

# Served through cors_proxy.py's `/local/` route (HTTP Range/HEAD support behind jupyter-server-proxy)
data = f"{PROXY_BASE}/local/{resolved_url_rel}"
print("Local GeoZarr URL:", data)

# Start the colour scale on the variable's value range (written by convert_zarr.py as actual_range)
_var_attrs = store_variables[variable].get("attributes", {})
_range = _var_attrs.get("valid_range") or _var_attrs.get("actual_range") or [0, 1]
style_min, style_max = float(_range[0]), float(_range[1])
print(f"Initial variable: {variable} | colour range [{style_min:.6g}, {style_max:.6g}]")

# ------------------------------------------------------------------------------
# Layer Configurations (EOxElements layer JSON, written as Python dicts)
# ------------------------------------------------------------------------------
layers = [
    {"type": "Tile", "source": {"type": "OSM"}},
    {
        "type": "WebGLTile",
        "properties": {
            "id": "local-geozarr",
            "title": resolved_store_path.stem,
            "layerControlExpand": True,
            "layerControlToolsExpand": True,
            "layerConfig": {
                "type": "style",
                "style": True,
                "autofill": True,
            },
        },
        "source": {
            "type": "GeoZarr",
            "crossOrigin": "anonymous",
            "url": data,
            "bands": [variable],
        },
        "style": {
            "variables": {
                "min": style_min,
                "max": style_max,
            },
            "color": [
                "interpolate",
                ["linear"],
                ["band", 1],
                ["var", "min"],
                ["color", 0, 210, 255, ["band", 2]],
                [
                    "+",
                    ["var", "min"],
                    ["*", ["-", ["var", "max"], ["var", "min"]], 0.3333333333333333],
                ],
                ["color", 254, 217, 118, ["band", 2]],
                [
                    "+",
                    ["var", "min"],
                    ["*", ["-", ["var", "max"], ["var", "min"]], 0.6666666666666666],
                ],
                ["color", 253, 141, 60, ["band", 2]],
                ["var", "max"],
                ["color", 189, 0, 38, ["band", 2]],
            ],
        },
    },
]

time_control_file = "./timecontrol.js"
with open(time_control_file, "r") as f:
    time_control_element = f.read()

if "eox-timecontrol-date" not in EOxTimecontrol._esm_template:
    EOxTimecontrol._esm_template = EOxTimecontrol._esm_template.replace(
        "view.el.appendChild(el);", time_control_element
    )

# ------------------------------------------------------------------------------
# Create and Display Map & Widgets
# ------------------------------------------------------------------------------
_xmin, _ymin, _xmax, _ymax = store_attrs["spatial:bbox"]
_center = (
    [(_xmin + _xmax) / 2, (_ymin + _ymax) / 2]
    if store_attrs["proj:code"] == "EPSG:4326"
    else [15, 45]
)

local_map = EOxMap(
    id="local-map",
    center=_center,
    zoom=4,
    layout=Layout(width="100%", height="500px"),
)
local_map.layers = layers

local_time_control = EOxTimecontrol(
    for_="#local-map",
    init_date=["first"],
    layout=Layout(width="100%"),
)

local_control = EOxLayercontrol(
    for_="#local-map",
    tools=["config", "opacity"],
    layout=Layout(flex="1", height="550px", width="300px", min_width="250px"),
)

local_map_container = VBox(
    [local_map, local_time_control],
    layout=Layout(flex="2", display="flex", flex_direction="column", gap="10px"),
)

HBox(
    [local_control, local_map_container],
    layout=Layout(width="100%", display="flex", gap="16px", align_items="flex-start"),
)

In [ ]:
# Stop the local CORS & Zarr proxy when you are done (the maps above stop loading data).
if "proxy" in globals() and proxy.poll() is None:
    proxy.terminate()
    print("Stopped proxy.")